# LSApp Time-Diff Preparation
This notebook loads LSApp event logs, computes interaction/session IDs, derives `open_time` and `close_time`, and saves processed outputs into `data/with_time_diff`.

In [24]:
%matplotlib inline

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)

In [25]:
# Fixed paths (no candidate search).
project_root = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
input_path = (project_root / 'data' / 'lsapp.tsv' / 'lsapp.tsv').resolve()

if not input_path.exists():
    raise FileNotFoundError(f'Could not find input file: {input_path}')

output_dir = (project_root / 'data' / 'with_time_diff').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Input file: {input_path}')
print(f'Output dir: {output_dir}')

Input file: D:\Git\context-aware-user-recommendation\data\lsapp.tsv\lsapp.tsv
Output dir: D:\Git\context-aware-user-recommendation\data\with_time_diff


In [26]:
df = pd.read_csv(input_path, sep='\t')
df = df.rename(columns={'lsapp.tsv': 'user_id'})

print(df.shape)
df.head()

(3658590, 5)


,user_id,session_id,timestamp,app_name,event_type
0,0.0,1.0,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0.0,1.0,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0.0,1.0,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0.0,1.0,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0.0,1.0,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened


In [27]:
# Basic type cleanup and ordering before ID construction.
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'user_id', 'app_name']).copy()
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# Optional helper subset for open events.
df_open = df.loc[df['event_type'] == 'Opened'].copy()

print('Clean rows:', len(df))
print('Opened rows:', len(df_open))

Clean rows: 3658589
Opened rows: 1673261


In [28]:
# Reproduce interaction/session logic similar to your reference notebook.
df['time_dff'] = df['timestamp'].diff()

is_new_interaction = (
    ((df['timestamp'] - df['timestamp'].shift(1) > pd.Timedelta(1, 'm')) & (df['event_type'] == 'Opened'))
    | ~(df['app_name'] == df['app_name'].shift(1))
    | ~(df['user_id'] == df['user_id'].shift(1))
)

is_new_session = (
    ((df['timestamp'] - df['timestamp'].shift(1) > pd.Timedelta(5, 'm')) & (df['event_type'] == 'Opened'))
    | ~(df['user_id'] == df['user_id'].shift(1))
)

df['interaction_id'] = is_new_interaction.cumsum()
df['session_id'] = is_new_session.cumsum()

In [47]:
# Create one row per interaction (same style as your reference notebook).
df_start = df.drop_duplicates(subset=['interaction_id'], keep='first').copy()
df_end = df.drop_duplicates(subset=['interaction_id'], keep='last').copy()

df_start = df_start.set_index('interaction_id')
df_end = df_end.set_index('interaction_id')

df_start['open_time'] = df_start['timestamp']
df_start['close_time'] = df_end['timestamp']
df_start['duration(s)'] = (df_start['close_time'] - df_start['open_time']).dt.total_seconds()

# Keep interaction-level dataset only (no repeated event rows).
df_interaction = df_start.reset_index().copy()

df_interaction.head(15)

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,duration(s)
0,1,0.0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaT,2018-01-16 06:01:05,2018-01-16 06:01:09,4.0
1,2,0.0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,33.0
2,3,0.0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,0.0
3,4,0.0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,5.0
4,5,0.0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,11.0
5,6,0.0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26,5.0
6,7,0.0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26,600.0
7,8,0.0,2,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58,2.0
8,9,0.0,2,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45,0.0
9,10,0.0,2,2018-01-16 07:20:46,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 07:20:46,2018-01-16 07:21:43,57.0


In [ ]:
# Filter out very short interactions (< 3 seconds).
df_interaction = df_interaction[df_interaction['duration(s)'] >= 3].copy()

print('Sessions after duration filter:', df_interaction['session_id'].nunique())
print('Rows after duration filter:', len(df_interaction))
df_interaction

Sessions after duration filter: 31376
Rows after duration filter: 209156


,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,duration(s)
0,1,0.0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaT,2018-01-16 06:01:05,2018-01-16 06:01:09,4.0
1,2,0.0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,33.0
2,4,0.0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,5.0
3,5,0.0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,11.0
4,6,0.0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26,5.0
...,...,...,...,...,...,...,...,...,...,...
209151,599626,291.0,33217,2018-04-06 12:05:33,Facebook,Opened,0 days 00:00:00,2018-04-06 12:05:33,2018-04-06 12:05:50,17.0
209152,599627,291.0,33217,2018-04-06 12:05:50,Facebook Messenger,Opened,0 days 00:00:00,2018-04-06 12:05:50,2018-04-06 12:18:51,781.0
209153,599630,291.0,33218,2018-04-06 12:37:57,Facebook Messenger,Opened,0 days 00:13:42,2018-04-06 12:37:57,2018-04-06 12:43:59,362.0
209154,599631,291.0,33218,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13,345.0


In [48]:
# Unique app_name list and count.
unique_apps = pd.Series(df['app_name'].dropna().astype(str).unique()).sort_values().reset_index(drop=True)
print('Number of unique app_name:', len(unique_apps))

# Preview
unique_apps.head(30)

Number of unique app_name: 87


0                         AOL
1             Amazon Shopping
2          Android In Call UI
3             Army Men Strike
4                       Badoo
5               Baseball Boy!
6               Brave Browser
7                  Calculator
8                    Calendar
9             Calorie Counter
10                     Camera
11               Clean Master
12                      Clock
13                   Contacts
14    DigiHUD Pro Speedometer
15                    Discord
16                EntertaiNow
17                   Facebook
18         Facebook Messenger
19                      Faceu
20                     Flickr
21         Flipboard Briefing
22                      Gmail
23                     Google
24              Google Chrome
25               Google Drive
26              Google Photos
27          Google Play Music
28          Google Play Store
29                   Hangouts
dtype: str

In [50]:
interaction_out = output_dir / 'lsapp_interactions_with_time_diff_filtered.tsv'

df_interaction.to_csv(interaction_out, sep='\t', index=False)

print(f'Saved interaction-level file: {interaction_out}')
print('interaction rows:', len(df_interaction))

Saved interaction-level file: D:\Git\context-aware-user-recommendation\data\with_time_diff\lsapp_interactions_with_time_diff_filtered.tsv
interaction rows: 599635
